# Transfer Additional US event to ds HUC10 Model 

#Goal 

Identify the us events that are driving the recurrence interval values at the tie-in and transfer those dss files to the downstream HUC, create new flow files and plans and then update the prj file to include the new information. 

In [1]:
import os
import re
import pathlib as pl

In [ ]:
os.chdir('..')
home = pl.Path(os.getcwd())

#user to set variables for the project. tagged as parameter for papermill runs
project = 'wy_fy23'
model_prefix = 'wy_bh_'
#target huc
target = '1008001302'

In [5]:
print('home is at: ',home)
home = pl.Path(home)
from src.hdf import *

inputs = home/'inputs'
outputs_base = home/'outputs'

export_folder = outputs_base/project/'trial_us_to_ds_events'
target_export = export_folder/str(model_prefix+target)

home is at:  c:\_code\hms_to_ras_sst


In [6]:
#location of upstream HUC dss files for connections between different HUC8s
#You must enter the folder where the hms dss files are stored for upstream HUCs
transfer_dss_location = inputs/project/'hms_ras'

In [7]:
#open dictionaries to understand which HUC the upstream one flows into and the downstream junction for each
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_into_dsJunction.json') as src:
    huc_connect_j = json.load(src)
with open(inputs/project/'dictionaries'/'junc_res_sink_next_junc_down.json') as src:
    j_to_j = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_Junctions.json') as src:
    huc_js = json.load(src)
with open(inputs/project/'dictionaries'/'source_if_reservoir.json') as src:
    src_js = json.load(src)
with open(inputs/project/'dictionaries'/'hms_feature_origin_dict.json') as src:
    src_huc_hms = json.load(src)
with open(inputs/project/'dictionaries'/'outofscope_HUC10_to_HUC10.json') as src:
    out_of_scope_huc_inflows = json.load(src)
with open(inputs/project/'dictionaries'/'Junction_Subbasins.json') as src:
    j_connect_sub = json.load(src)

In [8]:
#create folder for huc outputs
if not os.path.exists(target_export):
    os.makedirs(target_export)
    
#creating interior hydrology folder 
if not os.path.exists(target_export/'Hydrology'):
    os.mkdir(target_export/'Hydrology')

In [9]:
#this is the us huc desired to be selected. would attempt a list but a lot of pre-written code works for singular HUC
us_hucs = []
ds_j = []

##########make below a class/function later##########
#create HUC BC connections
bc_connections = {'junctions':{},'dss_path':{},'ts':{}}

for key, val in huc_connect_huc.items():
    if val == target:
        us_hucs.append(key)
        ds_j.append(huc_connect_j[key])
        bc_connections['junctions'][key] = huc_connect_j[key]
        bc_connections['dss_path'][key] = {}

In [48]:
bc_connections


{'junctions': {'1008001301': 'HUC_013_J_33'},
 'dss_path': {'1008001301': {}},
 'ts': {}}

In [49]:
events_dict_us = {}
#identify the controlling upstreamevent at the tie-in
for huc, j in  bc_connections['junctions'].items():
    events_dict_us[huc] = {j:{}}
    htmls = glob.glob(str(inputs/project/'hms_ras'/huc/'*_map.html'))
    events_dict_us = get_sst_storms_by_recurrence_us_huc(events_dict_us,huc,htmls,j)

In [50]:
events_dict_us

{'1008001301': {'HUC_013_J_33': {'0.002': 'P07_R-Y035-E0001',
   '0.01m': 'P10_R-Y057-E0001',
   '0.01p': 'P07_R-Y357-E0003',
   '0.01': 'P10_R-Y506-E0003',
   '0.02': 'P08_R-Y084-E0004',
   '0.04': 'P08_R-Y388-E0005',
   '0.1': 'P07_R-Y485-E0004'}}}

In [51]:
#Create dictionary with all the events required at to the upstream HUCs
complete_us_dict = {}

for huc in events_dict_us.keys():
    prob_shps = glob.glob(str(inputs/project/'hms_ras'/huc/'*junctions.geojson'))
    complete_us_dict.update(get_sst_storms_by_recurrence(huc,prob_shps))

In [52]:
complete_us_dict

{'1008001301': {'0.002': ['P05_R-Y198-E0002',
   'P01_R-Y048-E0001',
   'P07_R-Y035-E0001',
   'P01_R-Y465-E0002'],
  '0.01m': ['P10_R-Y057-E0001',
   'P05_R-Y199-E0002',
   'P09_R-Y300-E0003',
   'P02_R-Y251-E0001',
   'P05_R-Y426-E0002'],
  '0.01p': ['P07_R-Y019-E0002',
   'P01_R-Y377-E0002',
   'P07_R-Y357-E0003',
   'P01_R-Y465-E0002'],
  '0.01': ['P08_R-Y289-E0001',
   'P10_R-Y506-E0003',
   'P10_R-Y149-E0002',
   'P01_R-Y384-E0001',
   'P06_R-Y241-E0005'],
  '0.02': ['P08_R-Y084-E0004', 'P05_R-Y426-E0002', 'P09_R-Y029-E0001'],
  '0.04': ['P09_R-Y481-E0002', 'P08_R-Y388-E0005', 'P01_R-Y189-E0001'],
  '0.1': ['P07_R-Y485-E0004', 'P09_R-Y033-E0004', 'P02_R-Y376-E0002']}}

In [53]:
#attain a list of dss files already exported during the autobc process for the target huc
dss_basename_ds = []

dss_files_ds = glob.glob(str(outputs_base/project/f'{model_prefix+target}'/'[Hh]ydrology'/'*.dss'))
for path in dss_files_ds:
    path_base = os.path.basename(path).replace('_output.dss','')
    path_base_u = path_base[:4]+'R-'+path_base[-10:]
    dss_basename_ds.append(path_base_u)
    
print(len(dss_basename_ds),dss_basename_ds)

32 ['P01_R-Y121_E0003', 'P01_R-Y159_E0001', 'P02_R-Y376_E0002', 'P03_R-Y255_E0002', 'P03_R-Y298_E0002', 'P03_R-Y392_E0002', 'P03_R-Y483_E0001', 'P03_R-Y535_E0001', 'P04_R-Y058_E0001', 'P05_R-Y403_E0001', 'P05_R-Y426_E0002', 'P05_R-Y482_E0002', 'P06_R-Y275_E0001', 'P06_R-Y314_E0004', 'P06_R-Y359_E0001', 'P07_R-Y157_E0001', 'P07_R-Y187_E0002', 'P07_R-Y197_E0001', 'P07_R-Y202_E0002', 'P07_R-Y319_E0001', 'P08_R-Y487_E0001', 'P08_R-Y501_E0005', 'P09_R-Y138_E0003', 'P09_R-Y148_E0003', 'P09_R-Y154_E0003', 'P09_R-Y489_E0001', 'P10_R-Y147_E0002', 'P10_R-Y357_E0003', 'P10_R-Y460_E0001', 'P10_R-Y491_E0002', 'P10_R-Y492_E0001', 'P10_R-Y507_E0003']


In [54]:
#create a dictionary with available dss files from the autobc output for each of the upstream HUCs
huc_dss_base = {}
#the paths for all of the dss files available for all upstream HUCs
dss_files_tot = set()

for huc in events_dict_us.keys():
    dss_files = glob.glob(str(outputs_base/project/f'{model_prefix+huc}'/'[Hh]ydrology'/'*.dss'))
    dss_basenames = []
    for path in dss_files:
        dss_files_tot.add(path)
        us_basename_u = os.path.basename(path).replace('_output.dss','')
        dss_basenames.append(us_basename_u[:4]+'R-'+us_basename_u[-10:])
    huc_dss_base.update({huc: dss_basenames})
    print(len(dss_files),f': amount of dss files identified for huc {huc}')

25 : amount of dss files identified for huc 1008001301


In [55]:
len(dss_files_tot)

25

In [56]:
#adding events from us dictionary if not already in the downstream model export
events_req = set()
for huc in events_dict_us.keys():
    for junc in events_dict_us[huc].keys():
        for rec, event in events_dict_us[huc][junc].items():
            if event not in dss_basename_ds:
                events_req.add(event)
            else:
                print(f'event identified already in target huc {target} Hydrology folder:', event)
                pass
print("Required events =",events_req, len(events_req))

Required events = {'P08_R-Y084-E0004', 'P07_R-Y485-E0004', 'P07_R-Y035-E0001', 'P10_R-Y506-E0003', 'P07_R-Y357-E0003', 'P08_R-Y388-E0005', 'P10_R-Y057-E0001'} 7


In [58]:
events_req

{'P07_R-Y035-E0001',
 'P07_R-Y357-E0003',
 'P07_R-Y485-E0004',
 'P08_R-Y084-E0004',
 'P08_R-Y388-E0005',
 'P10_R-Y057-E0001',
 'P10_R-Y506-E0003'}

In [ ]:
######## IN PROGRESS - PENDING TO CONTINUE BELOW. NEED TO ADD 10% DIFFERENCE CHECK BETWEEN CURRENT EVENTS AND UPSTREAM EVENTS. 
######## IT IS LIKELY THAT THIS NOTEBOOK WILL NEED TO STAY

### Copying over dss files not already in downstream model

In [24]:
dss_files_model = []
for event in events_req:
    evnt_name = re.search('Y\d+-E\d+',event).group()
    evnt_name_u = evnt_name.replace('-','_')
    criteria = f'\S+(R\d+-)?{evnt_name_u}_output.dss'
    r = re.compile(criteria)
    dss_matches = list(filter(r.match, dss_files_tot))
    # print(dss_matches)
    if len(dss_matches) > 1:
        evnt_name_full = re.search('R\d+-Y\d+-E\d+',event).group()
        evnt_name_full_u = evnt_name_full.replace('-','_')
        criteria = f'\S+{evnt_name_full_u}_output.dss' 
        r = re.compile(criteria)
        dss_matches = list(filter(r.match, dss_matches))
    assert len(dss_matches) != 0, "there are no dss files matches"
    # assert len(dss_matches) == 1, "there are more than 1 dss files with the same Y and E number that don't have a R number specified"
    out_name = event.replace('-','_')+"_output.dss"
    
    #quick check to see if the event being copied over is from the same HUC8 as our target. This is a precaution to ensure the junction information will be included in the unsteady flow information copied over 
    if f'wy_gdg_{target[:8]}' not in dss_matches[0]:
        print('dss file is being copied from a HUC10 in a different HUC8: Event ', event)
        from_huc10_base = re.search(r'\\wy_gdg_1404......\\', dss_matches[0])
        from_huc8 = str(from_huc10_base.group())[8:-3]
        
        event_temp = event.replace('-','_')
        # event_mod = event_[:1]+'-'+event_[1:]
        print(f'attempting to pull dss file {event_temp} from {from_huc8}')
        
        shutil.copy(transfer_dss_location/target[:8]/f'{event_temp}_output.dss',target_export/'Hydrology'/out_name)
        dss_files_model.append(str(target_export/'Hydrology'/out_name))
    else:
        print(f'copying existing dss file {dss_matches[0]} for event {event}from auto-bc creation folder')
        shutil.copy(dss_matches[0],target_export/'Hydrology'/out_name)
        dss_files_model.append(str(target_export/'Hydrology'/out_name))

dss file is being copied from a HUC10 in a different HUC8: Event  R3-Y254-E0001
attempting to pull dss file R3_Y254_E0001 from 14040101
copying existing dss file \\us0236-ppfss01\shared_projects\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\wy_gdg_1404010305\Hydrology\R4_Y220_E0003_output.dss for event R4-Y220-E0003from auto-bc creation folder
dss file is being copied from a HUC10 in a different HUC8: Event  R6-Y406-E0005
attempting to pull dss file R6_Y406_E0005 from 14040101
copying existing dss file \\us0236-ppfss01\shared_projects\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\wy_gdg_1404010305\Hydrology\R6_Y053_E0001_output.dss for event R6-Y053-E0001from auto-bc creation folder
copying existing dss file \\us0236-ppfss01\shared_projects\173432208011\studies\1_great_divide_green_watershed\product

In [25]:
dss_files_model

['\\\\us0236-ppfss01\\shared_projects\\173432208011\\studies\\1_great_divide_green_watershed\\production\\engineering_riverine\\hydraulics\\hydrology_incorporation\\_code\\outputs\\wy_fy22\\trial_us_to_ds_events\\wy_gdg_1404010306\\Hydrology\\R3_Y254_E0001_output.dss',
 '\\\\us0236-ppfss01\\shared_projects\\173432208011\\studies\\1_great_divide_green_watershed\\production\\engineering_riverine\\hydraulics\\hydrology_incorporation\\_code\\outputs\\wy_fy22\\trial_us_to_ds_events\\wy_gdg_1404010306\\Hydrology\\R4_Y220_E0003_output.dss',
 '\\\\us0236-ppfss01\\shared_projects\\173432208011\\studies\\1_great_divide_green_watershed\\production\\engineering_riverine\\hydraulics\\hydrology_incorporation\\_code\\outputs\\wy_fy22\\trial_us_to_ds_events\\wy_gdg_1404010306\\Hydrology\\R6_Y406_E0005_output.dss',
 '\\\\us0236-ppfss01\\shared_projects\\173432208011\\studies\\1_great_divide_green_watershed\\production\\engineering_riverine\\hydraulics\\hydrology_incorporation\\_code\\outputs\\wy_fy22\\

In [26]:
#copy over the project file for the target huc from the autobc export 
if not os.path.exists(target_export/f'wy_gdg_{target}.prj'):
        prj_out_name = shutil.copy(outputs_base/project/f'wy_gdg_{target}'/f'wy_gdg_{target}.prj',target_export/f'wy_gdg_{target}.prj')
else:
        prj_out_name = target_export/f'wy_gdg_{target}.prj'

### Copying over Geometry files

In [27]:
model_name = f'wy_gdg_{target}'

geo_out_hdf = shutil.copy(outputs_base/project/model_name/f'{model_name}.g51.hdf',target_export/f'{model_name}.g51.hdf')
geo_out = shutil.copy(outputs_base/project/model_name/f'{model_name}.g51',target_export/f'{model_name}.g51')

In [28]:
hf_geo = h5py.File(str(geo_out_hdf),'r')

#domain names
domain_geo = str(list(hf_geo['Geometry']['2D Flow Areas']['Attributes'])[0][0]).strip("'b\'")

In [29]:
plan_path, in_geom_path, in_flow_path = get_current_ras_files(outputs_base/project/model_name/f'{model_name}.prj')

## Update the files from AutoBC creation output 

In [30]:
## Once we have the new plans for tie-ins we need to add them to 
# the existing plan and flow files and then update the prj file

#get max plan value as start_id
start_id = 51
plan_files = glob.glob(str(plan_path.parent/'*.p*[!rj][!.hdf]'))
for p in plan_files:
    p_num = re.search('.p[\d]+',p)
    if int(p_num.group()[2:]) >= start_id:
        start_id += 1

In [31]:
start_id

69

### Add in additional dss information per auto BC creation workflow

In [32]:
outputs = outputs_base/project/model_name

In [33]:
## ext bc line info
if os.path.exists(f'{outputs}/{target}_ext_forcing_bc_lines.shp'):
    ext_bcs = gpd.read_file(f'{outputs}/{target}_ext_forcing_bc_lines.shp')
else:
    ext_bcs = pd.DataFrame(columns=['empty'])

ext_bc_dict = ext_bcs.to_dict()
ext_index = ext_bcs.index.tolist()
ext_bc_dict['in_flow_path'] = in_flow_path
ext_bc_dict['in_plan_path'] = plan_path

In [34]:
## ext bc oos line info
if os.path.exists(f'{outputs}/{target}_ext_forcing_bc_lines_oos.shp'):
    ext_bcs_oos = gpd.read_file(f'{outputs}/{target}_ext_forcing_bc_lines_oos.shp')
    ext_bcs_combined = pd.concat([ext_bcs, ext_bcs_oos])
    ext_bcs_combined.reset_index(inplace=True,drop=True)
    ext_bc_dict_combined = ext_bcs_combined.to_dict()
    ext_index_combined = ext_bcs_combined.index.tolist()
else:
    ext_bc_dict_oos = {}
    ext_index_oos = []
    ext_bc_dict_combined = ext_bc_dict
    ext_index_combined = ext_index


ext_bc_dict_combined['in_flow_path'] = in_flow_path
ext_bc_dict_combined['in_plan_path'] = plan_path

In [35]:
## int bc line info
if os.path.exists(f'{outputs}/{target}_in_forcing_bc_lines.shp'):
    int_bcs = gpd.read_file(f'{outputs}/{target}_in_forcing_bc_lines.shp')
else:
    int_bcs = pd.DataFrame(columns=['empty'])

int_bc_dict = int_bcs.to_dict()
int_index = int_bcs.index.tolist()
int_bc_dict['in_flow_path'] = in_flow_path
int_bc_dict['in_plan_path'] = plan_path

In [36]:
## flow reduction reaches
# if os.path.exists(f'{outputs}/{target}_flow_reduction_forcing_bc_lines.shp'):
#     reaches_r_s = gpd.read_file(f'{outputs}/{target}_flow_reduction_forcing_bc_lines.shp')
# else:
#     reaches_r_s = pd.DataFrame(columns=['empty'])
    
# int_r_bc_dict = reaches_r_s.to_dict()
# int_r_index = reaches_r_s.index.tolist()
# int_r_bc_dict['in_flow_path'] = in_flow_path
# int_r_bc_dict['in_plan_path'] = plan_path
#get hms schematic information

schem_folder = inputs/project/target[4:8]/f'HUC{target[:8]}_SST_schematic'
if os.path.exists(schem_folder/'hms_plotting'):
    schem_folder = schem_folder/'hms_plotting'
# junctions = gpd.read_file(str(schem_folder/f'HUC{huc[:8]}_SST_Junction.shp'))
# sinks = gpd.read_file(str(schem_folder/f'HUC{huc[:8]}_SST_Sink.shp'))
# if os.path.exists(str(schem_folder/f'HUC{huc[:8]}_SST_Reservoir.shp')):
#     reservoirs = gpd.read_file(str(schem_folder/f'HUC{huc[:8]}_SST_Reservoir.shp'))
#     hms_points = pd.concat([junctions, sinks,reservoirs]).set_index('index')
# else:
#     hms_points = pd.concat([junctions, sinks]).set_index('index')
# subbasins = gpd.read_file(str(schem_folder/f'HUC{huc[:8]}_SST_Subbasin.shp'))
reaches = gpd.read_file(str(schem_folder/f'HUC{target[:8]}_SST_Reach.shp'))




#get reach junction dictionary
with open(inputs/project/'dictionaries'/'reachfromjunction.json') as src:
    reach_to_us_junction = json.load(src)
##filter to relevant flow reduction reaches\
reaches_col = reaches.columns.to_list()
if 'Channel Lo' in reaches_col:
    cl = 'Channel Lo'
    sb = 'Channel _1'
    rd = 'Channel _2'
else:
    cl = 'Channel _1'
    sb = 'Channel _2'
    rd = 'Channel _3'
flow_r_filter = reaches[cl] == 'Constant'
#channel 2 is initial flow reduction, channel 3 is % reduction of flow hydrograph so adjusted flow hydrograph = (inflow - 5) *.8
reaches_r = reaches.loc[flow_r_filter]
#filter by relevant huc
if target[:8] == '14040102':
    ds = 'downstream'
else:
    ds = 'Downstream'
reaches_r_s = reaches_r.copy().loc[reaches_r[ds].isin(huc_js[target]) == True]
if not reaches_r_s.empty:
    reaches_r_s['subtraction'] = reaches_r_s[sb]
    reaches_r_s['reduction'] = reaches_r_s[rd]
else:
    reaches_r_s['subtraction'] = 0
    reaches_r_s['reduction'] = 0
    
reaches_r_s['upstream'] = reaches_r_s['name'].apply(lambda x: reach_to_us_junction[x])

####modify to make useful for bc purposes###
#to be consistent with ext code
reaches_r_s['junction'] = reaches_r_s.name
#Internal BC EG slope should only impact the normal depth calc used to distribute flow to adjacent cells. 
#Assume uniform value of 0.01 (relatively steep) to force flow to the low point of the cell
reaches_r_s['slope'] = 0.01

reaches_r_s.reset_index(inplace=True)

int_r_bc_dict = reaches_r_s.to_dict()
int_r_index = reaches_r_s.index.tolist()
int_r_bc_dict['in_flow_path'] = in_flow_path
int_r_bc_dict['in_plan_path'] = plan_path

In [37]:
## flow sources
if os.path.exists(f'{outputs}/{target}_in_forcing_bc_lines_sources.shp'):
    src_bcs = gpd.read_file(f'{outputs}/{target}_in_forcing_bc_lines_sources.shp')
else:
    src_bcs = pd.DataFrame(columns=['empty'])
src_bc_dict = src_bcs.to_dict()
src_index = src_bcs.index.tolist()
src_bc_dict['in_flow_path'] = in_flow_path
src_bc_dict['in_plan_path'] = plan_path

######################### Latest Additions

In [38]:
#plans and unsteady flow files are not recognized over 100 by the models in naming. Add clause to resolve if added dss_files are over

if len(dss_files_model)+start_id <= 100:
    dss_matches = write_updated_ext_bc_files(domain_geo,dss_files_model,ext_index_combined,ext_bc_dict_combined,int_index,int_bc_dict,int_r_index,int_r_bc_dict,src_index, src_bc_dict, inputs,target_export,project,target,start_id,src_huc_hms,append=True)

else:
    fixed_id_temp = 100 - start_id
    dss_matches = write_updated_ext_bc_files(domain_geo,dss_files_model[0:fixed_id_temp],ext_index_combined,ext_bc_dict_combined,int_index,int_bc_dict,int_r_index,int_r_bc_dict,src_index, src_bc_dict, inputs,target_export,project,target,start_id,src_huc_hms,append=True)
    
    new_start_id = 51 - (len(dss_files_model)-fixed_id_temp)
    dss_matches = write_updated_ext_bc_files(domain_geo,dss_files_model[fixed_id_temp:],ext_index_combined,ext_bc_dict_combined,int_index,int_bc_dict,int_r_index,int_r_bc_dict,src_index, src_bc_dict, inputs,target_export,project,target,new_start_id,src_huc_hms,append=True)

print('\nCompleted')

us_huc8_triggered_external_junction 14040101_HUC_101_J_464
us_huc8_triggered_external_junction 14040104_HUC_104_J_1
us_huc8_triggered_external_junction 14040101_HUC_101_J_464
us_huc8_triggered_external_junction 14040104_HUC_104_J_1
us_huc8_triggered_external_junction 14040101_HUC_101_J_464
us_huc8_triggered_external_junction 14040104_HUC_104_J_1
us_huc8_triggered_external_junction 14040101_HUC_101_J_464
us_huc8_triggered_external_junction 14040104_HUC_104_J_1
us_huc8_triggered_external_junction 14040101_HUC_101_J_464
us_huc8_triggered_external_junction 14040104_HUC_104_J_1
us_huc8_triggered_external_junction 14040101_HUC_101_J_464
us_huc8_triggered_external_junction 14040104_HUC_104_J_1
us_huc8_triggered_external_junction 14040101_HUC_101_J_464
us_huc8_triggered_external_junction 14040104_HUC_104_J_1
us_huc8_triggered_external_junction 14040101_HUC_101_J_464
us_huc8_triggered_external_junction 14040104_HUC_104_J_1
us_huc8_triggered_external_junction 14040101_HUC_101_J_464
us_huc8_trigg

In [39]:
#Complete